<a href="https://colab.research.google.com/github/bogdangit-art/task_modul_4/blob/main/notebooks/train_model_task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## In this notebook we will train the selected machine learning model from the datacleaning notebook.

##1. Loading python libraries.

In [10]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

##2. Data loading.

In [11]:
url = "https://raw.githubusercontent.com/bogdangit-art/task_modul_4/main/data/products.csv"
df = pd.read_csv(url)

print(f"Original number of rows: {len(df)}")
print(f"Original number of columns: {len(df.columns)}")


Original number of rows: 35311
Original number of columns: 8


##3. Cleaning column names.

In [12]:
df.columns = df.columns.str.strip()

print("Columns:")
print(df.columns.tolist())

Columns:
['product ID', 'Product Title', 'Merchant ID', 'Category Label', '_Product Code', 'Number_of_Views', 'Merchant Rating', 'Listing Date']


##4. Cleaning product titles column.

In [13]:
# Convert Product Title to string
df["Product Title"] = df["Product Title"].astype("string")

# Remove leading/trailing spaces
df["Product Title"] = df["Product Title"].str.strip()

# Replace multiple spaces with one space
df["Product Title"] = df["Product Title"].str.replace(
    r"\s+",
    " ",
    regex=True
)

# Remove missing titles
df = df.dropna(subset=["Product Title"])

# Remove empty titles
df = df[df["Product Title"] != ""]

print(f"Rows after title cleaning: {len(df)}")

Rows after title cleaning: 35139


##5. Cleaning category labels column.

In [14]:
# Convert category to string
df["Category Label"] = df["Category Label"].astype("string")

# Remove leading/trailing spaces
df["Category Label"] = df["Category Label"].str.strip()

# Remove missing categories
df = df.dropna(subset=["Category Label"])

# Remove empty categories
df = df[df["Category Label"] != ""]

##6. Standardizing category labels column.

In [15]:
category_mapping = {
    "CPU": "CPUs",
    "Mobile Phone": "Mobile Phones",
    "fridge": "Fridges"
}

df["Category Label"] = df["Category Label"].replace(
    category_mapping
)

print("\nFinal category distribution:")
print(df["Category Label"].value_counts())


Final category distribution:
Category Label
Fridge Freezers     5470
Mobile Phones       4057
Washing Machines    4015
CPUs                3831
Fridges             3559
TVs                 3541
Dishwashers         3405
Digital Cameras     2689
Microwaves          2328
Freezers            2201
Name: count, dtype: Int64


##7. Duplicate removal.

In [16]:
before_duplicates = len(df)

df = df.drop_duplicates()

after_duplicates = len(df)

print(f"Rows before duplicates removal: {before_duplicates}")
print(f"Rows after duplicates removal:  {after_duplicates}")
print(f"Duplicates removed:             {before_duplicates - after_duplicates}")

Rows before duplicates removal: 35096
Rows after duplicates removal:  35096
Duplicates removed:             0


##8. Preparing data for machine learning.

In [17]:
# Input feature
X = df["Product Title"]

# Target variable
y = df["Category Label"]

print(f"Number of samples: {len(X)}")
print(f"Number of categories: {y.nunique()}")

print("\nCategories:")
print(sorted(y.unique()))

Number of samples: 35096
Number of categories: 10

Categories:
['CPUs', 'Digital Cameras', 'Dishwashers', 'Freezers', 'Fridge Freezers', 'Fridges', 'Microwaves', 'Mobile Phones', 'TVs', 'Washing Machines']


##9. Train/Test splitting the data.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

Training samples: 28076
Testing samples:  7020


##10. Creating Linear SVM Model Pipeline.

In [19]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LinearSVC()
    )
])

print("Model: TF-IDF + Linear SVM")

Model: TF-IDF + Linear SVM


##11. Training the model.

In [20]:
model.fit(X_train, y_train)

print("Training completed successfully!")

Training completed successfully!


##12. Evaluating the model.

In [21]:
predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)


Accuracy: 0.9650
Accuracy: 96.50%

Classification Report:
                  precision    recall  f1-score   support

            CPUs       1.00      1.00      1.00       766
 Digital Cameras       1.00      0.99      1.00       538
     Dishwashers       0.95      0.93      0.94       681
        Freezers       0.97      0.95      0.96       440
 Fridge Freezers       0.92      0.97      0.94      1094
         Fridges       0.93      0.92      0.93       712
      Microwaves       0.98      0.97      0.98       466
   Mobile Phones       0.98      0.99      0.98       812
             TVs       0.98      0.99      0.99       708
Washing Machines       0.97      0.94      0.95       803

        accuracy                           0.96      7020
       macro avg       0.97      0.96      0.97      7020
    weighted avg       0.97      0.96      0.96      7020



##13. Saving the model.
Prin acest script se salveaza local fisierul pkl, pe care il voi uploada in github separat.

In [ ]:
#model_filename = "product_category_model.pkl"

#joblib.dump(model,model_filename)

#print(f"Model saved successfully as: {model_filename}")